In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier



df = pd.read_csv('./Titanic-Dataset.csv')

df.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [4]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [5]:
df["Embarked"].value_counts()

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [6]:
# taking independent and dependent features
X = df.drop(columns=['Survived'])
y = df['Survived']

X.head()
# y.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S


In [7]:
# testing and training data split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# print(X_train.shape)
# print(X_test.shape)
# print(y_train.shape)
# print(y_test.shape)

In [8]:
# applying Simple Imputer Over here, for missing values


si_age = SimpleImputer()
si_embarked = SimpleImputer(strategy='most_frequent')


X_train_age = si_age.fit_transform(X_train[['Age']])
X_train_Embarked = si_embarked.fit_transform(X_train[['Embarked']])

X_test_age = si_age.transform(X_test[['Age']])
X_test_Embarked = si_embarked.transform(X_test[['Embarked']])

In [17]:
# X_train_age_df = pd.DataFrame(X_train_age, columns=['Age'])
# X_train_age_df.isnull().sum()

# X_train_age_df

In [22]:
# applying OneHotEncoder Over here


ohe_sex = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_embarked = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

X_train_sex = ohe_sex.fit_transform(X_train[['Sex']])
X_train_embarked = ohe_embarked.fit_transform(X_train[['Embarked']])

X_test_sex = ohe_sex.transform(X_test[['Sex']])
X_test_embarked = ohe_embarked.transform(X_test[['Embarked']])


 


# X_train['Age'] = X_train_age
# X_train['Embarked'] = X_train_Embarked


# X_test['Age'] = X_test_age
# X_test['Embarked'] = X_test_Embarked



In [21]:
# ohe_sex.get_feature_names_out(['Sex'])
X_train_embarked

array([[0., 0., 1., 0.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.],
       ...,
       [0., 0., 1., 0.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.]])

In [23]:
# X_train_sex_df = pd.DataFrame(X_train_sex, columns=ohe_sex.get_feature_names_out(['Sex']))

# X_train_embarked_df = pd.DataFrame(
#     X_train_embarked,
#     columns=ohe_embarked.get_feature_names_out(['Embarked'])
# )
# # X_test_sex_df = pd.DataFrame(X_test_sex, columns=['male', 'female'])
# # X_test_embarked_df = pd.DataFrame(X_test_embarked, columns=['S', 'C', 'Q'])


X_train_remaining_features = X_train.drop(columns=['Sex', 'Age', 'Embarked'])


X_test_remaining_features = X_test.drop(columns=['Sex', 'Age', 'Embarked'])

In [24]:
# X_train_sex_df
X_train_remaining_features

,Pclass,SibSp,Parch,Fare
331,1,0,0,28.5000
733,2,0,0,13.0000
382,3,0,0,7.9250
704,3,1,0,7.8542
813,3,4,2,31.2750
...,...,...,...,...
106,3,0,0,7.6500
270,1,0,0,31.0000
860,3,2,0,14.1083
435,1,1,2,120.0000


In [25]:
X_train_transformed = np.concatenate(
    (
        X_train_age,
        X_train_sex,
        X_train_embarked,
        X_train_remaining_features,
    ),
    axis=1
)
X_test_transformed = np.concatenate(
    (
        X_test_age,
        X_test_sex,
        X_test_embarked,
        X_test_remaining_features,
    ),
    axis=1
)

In [26]:
from sklearn.metrics import accuracy_score


clf = DecisionTreeClassifier()


clf.fit(X_train_transformed, y_train)

y_pred = clf.predict(X_test_transformed)

acc = accuracy_score(y_test, y_pred)

print(f"Accuracy: {acc*100}")

Accuracy: 78.2122905027933


In [30]:
import pickle

pickle.dump(ohe_sex, open('model/ohe_sex.pkl', 'wb'))
pickle.dump(ohe_embarked, open('model/ohe_embarked.pkl', 'wb'))


pickle.dump(clf, open('model/model.pkl', 'wb'))

In [28]:
print(model)


DecisionTreeClassifier()
